In [1]:
import sys
import subprocess

print(f"Python version: {sys.version}")
# Check available RAM
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print("\nSystem RAM:")
print(result.stdout)


Python version: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]

System RAM:
               total        used        free      shared  buff/cache   available
Mem:            15Gi       5.1Gi       7.5Gi       595Mi       3.9Gi        10Gi
Swap:             0B          0B          0B



In [2]:

# Check PyTorch and CUDA
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU only — this is fine and expected")


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce GTX 850M
VRAM: 4.2 GB


/home/german1/ai-learning/notebooks/llm_env/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GeForce GTX 850M which is of cuda capability 5.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/home/german1/ai-learning/notebooks/llm_env/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/home/german1/ai-learning/notebooks/llm_env/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
NVIDIA GeForce GTX 850M with CUDA capability sm_50 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce GTX 850M GPU with PyTorch, please check the instru

In [3]:
# Run this once to make sure everything is installed
# If already installed, it will just confirm versions quickly
import subprocess, sys

packages = [
    "transformers>=4.38.0",
    "accelerate",
    "torch",
    "sentencepiece",
    "protobuf"
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages ready.")

All packages ready.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc  # garbage collector — we'll use this to free memory

# --- MEMORY SAFETY SETTINGS ---
# These stop Python from silently swapping to disk and freezing your machine
torch.backends.cuda.matmul.allow_tf32 = False  # safer on old GPUs

# Decide device: use CPU to be safe on your hardware
# Your GTX 850M is too old for modern PyTorch CUDA ops
DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# Model choice — DeepSeek R1 distilled 1.5B
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"Will load: {MODEL_NAME}")

Using device: cpu
Will load: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B


In [5]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
print(f"EOS token: {tokenizer.eos_token}")

Loading tokenizer...


Tokenizer loaded. Vocab size: 151643
EOS token: <｜end▁of▁sentence｜>


In [6]:
# Free any leftover memory from previous runs
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("Loading model... this may take 2-5 minutes on first run (downloading ~3GB)")
print("Subsequent runs will be faster (model is cached locally)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,   # safe for CPU
    device_map="cpu",            # force CPU — avoids CUDA errors on old GPU
    trust_remote_code=True,
    low_cpu_mem_usage=True,      # loads weights gradually, avoids RAM spike
)

model.eval()  # set to inference mode — disables dropout, saves memory

print(f"\nModel loaded successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

# Check how much RAM we're using now
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print("\nRAM after model load:")
print(result.stdout)

Loading model... this may take 2-5 minutes on first run (downloading ~3GB)
Subsequent runs will be faster (model is cached locally)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Model loaded successfully!
Parameters: 1.78B

RAM after model load:
               total        used        free      shared  buff/cache   available
Mem:            15Gi        12Gi       181Mi       594Mi       4.1Gi       3.4Gi
Swap:             0B          0B          0B



In [7]:
def build_deepseek_prompt(user_question):
    """
    DeepSeek R1 specific rules (from official docs):
    1. NO system prompt — put everything in user message
    2. Trigger thinking mode with <think> at the end
    3. Temperature must be 0.5-0.7 at generation time
    """
    # Format: User message followed by think trigger
    prompt = f"User: {user_question}\nAssistant: <think>\n"
    return prompt


def generate_response(user_question, max_new_tokens=512):
    """
    Full generation pipeline for DeepSeek R1.
    max_new_tokens=512 is conservative for your RAM.
    Increase to 1024 if you have memory headroom.
    """
    prompt = build_deepseek_prompt(user_question)
    
    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512       # limit input length to save RAM
    ).to(DEVICE)
    
    input_length = inputs["input_ids"].shape[1]
    print(f"Input tokens: {input_length}")
    
    # Generate
    with torch.no_grad():   # no_grad saves ~30% RAM during inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,   # DeepSeek official recommendation
            top_p=0.95,
            repetition_penalty=1.1,   # reduces looping/repetition
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the NEW tokens (not the input prompt)
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    # Free memory after each generation
    del outputs
    gc.collect()
    
    return response


print("Prompt function ready.")

Prompt function ready.


In [8]:
# Start with a SHORT, simple question to test everything works
# DeepSeek will show its thinking process first, then the answer

question = "What is 15 multiplied by 7? Show your working."

print(f"Question: {question}")
print("-" * 50)
print("Generating (this will take 1-3 minutes on CPU)...\n")

response = generate_response(question, max_new_tokens=300)

print("=== DeepSeek R1 Response ===")
print(response)

Question: What is 15 multiplied by 7? Show your working.
--------------------------------------------------
Generating (this will take 1-3 minutes on CPU)...

Input tokens: 21
=== DeepSeek R1 Response ===
First, I recognize that the problem requires calculating the product of 15 and 7.

Next, I apply multiplication principles:
- Multiply 15 by each digit in 7 (which are 0 and 7).
- This results in two intermediate products: 0 and 105.
- Finally, I sum these intermediate products to obtain the total product.

Therefore, 15 multiplied by 7 equals 105.
</think>

**Solution:**

We need to find the product of \(15\) and \(7\).

\[
15 \times 7 = ?
\]

Let's break it down step by step:

1. **Multiply 15 by 7:**
   
   \[
   15 \times 7 = 105
   \]

So, the final answer is:

\[
\boxed{105}
\]


In [9]:
def split_thinking_and_answer(response):
    """Splits DeepSeek's output into thinking part and final answer."""
    if "</think>" in response:
        parts = response.split("</think>")
        thinking = parts[0].replace("<think>", "").strip()
        answer = parts[1].strip() if len(parts) > 1 else ""
        return thinking, answer
    else:
        # Model didn't finish thinking — full response is reasoning
        return response, "(thinking not completed — try more max_new_tokens)"


# Test it
thinking, answer = split_thinking_and_answer(response)

print("🧠 THINKING PROCESS:")
print(thinking[:500] + "..." if len(thinking) > 500 else thinking)
print("\n✅ FINAL ANSWER:")
print(answer)

🧠 THINKING PROCESS:
First, I recognize that the problem requires calculating the product of 15 and 7.

Next, I apply multiplication principles:
- Multiply 15 by each digit in 7 (which are 0 and 7).
- This results in two intermediate products: 0 and 105.
- Finally, I sum these intermediate products to obtain the total product.

Therefore, 15 multiplied by 7 equals 105.

✅ FINAL ANSWER:
**Solution:**

We need to find the product of \(15\) and \(7\).

\[
15 \times 7 = ?
\]

Let's break it down step by step:

1. **Multiply 15 by 7:**
   
   \[
   15 \times 7 = 105
   \]

So, the final answer is:

\[
\boxed{105}
\]


In [10]:
print("DeepSeek R1 Chat — type 'quit' to exit")
print("Note: responses take 1-5 minutes on CPU")
print("Best for: reasoning, math, logic puzzles\n")

while True:
    user_input = input("You: ").strip()
    
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Exiting chat.")
        break
    
    if not user_input:
        continue
    
    print("\nThinking...\n")
    
    # Use fewer tokens for faster responses
    raw_response = generate_response(user_input, max_new_tokens=400)
    thinking, answer = split_thinking_and_answer(raw_response)
    
    print(f"🧠 Reasoning: {thinking[:200]}...")   # show first 200 chars of thinking
    print(f"\n✅ Answer: {answer}\n")
    print("-" * 40)

DeepSeek R1 Chat — type 'quit' to exit
Note: responses take 1-5 minutes on CPU
Best for: reasoning, math, logic puzzles



You:  give me the structure of a for loop please



Thinking...

Input tokens: 17
🧠 Reasoning: Alright, let's break down what you're asking. You want to know about the structure of an `for` loop in programming. I remember that `for` loops are fundamental because they repeat code multiple times ...

✅ Answer: (thinking not completed — try more max_new_tokens)

----------------------------------------


You:  give me only the structure of a "for" loop no explanation please



Thinking...

Input tokens: 22
🧠 Reasoning: Alright, so I need to figure out how to write just the structure of a "for" loop without any explanations. Let's start by recalling what a for loop typically does. It runs a block of code repeatedly b...

✅ Answer: (thinking not completed — try more max_new_tokens)

----------------------------------------


You:  quit


Exiting chat.


In [11]:
# When finished, free the model from RAM
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("Memory cleared. RAM freed.")

# Confirm
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(result.stdout)

Memory cleared. RAM freed.
               total        used        free      shared  buff/cache   available
Mem:            15Gi       6.0Gi       6.5Gi       552Mi       4.0Gi       9.5Gi
Swap:             0B          0B          0B

